In [1]:
import pandas as pd
import numpy as np
import re 

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence 
from deep_translator import GoogleTranslator

In [2]:
import torch
import numpy as np
import random

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [ ]:
import re
from collections import Counter

# df = pd.read_csv('phishing_emails.csv')
# df.drop_duplicates(subset=['text'], keep='first', inplace=True)
# df = df.sample(n=50000, random_state=42).reset_index(drop=True)
# df_extended = pd.read_csv('extra_data_cleaned.csv')
# df_extended.drop_duplicates(subset=['text'], keep='first', inplace=True)
# # df_extended = df_extended.sample(n=4000, random_state=42).reset_index(drop=True)
# df_add_train = pd.read_csv('train_ex_data.csv')
# df_add_train.drop_duplicates(subset=['text'], keep='first', inplace=True)
# df_ham_1000 = pd.read_csv('modern_ham_1000.csv')
# df_ham_1000.drop_duplicates(subset=['text'], keep='first', inplace=True)
# df_new_1000 = pd.read_csv('new_1000.csv')
# df_new_1000.drop_duplicates(subset=['text'], keep='first', inplace=True)
# df_old = pd.read_csv('spam_Emails_data.csv')
# df_old.drop_duplicates(subset=['text'], keep='first', inplace=True)
# # df_old = df_old.sample(n=500, random_state=42).reset_index(drop=True)
# df_old1 = df_old[df_old['label'] == 'Ham'].sample(n=500, random_state=42).reset_index(drop=True)
# df_old2 = df_old[df_old['label'] == 'Spam'].sample(n=60, random_state=42).reset_index(drop=True)
# df_old = pd.concat([df_old1, df_old2], ignore_index=True)
# df_old['label'] = df_old['label'].map({'Ham': 0, 'Spam': 1})
# df_add_test = pd.read_csv('test_ex_data.csv')
# df_add_test.drop_duplicates(subset=['text'], keep='first', inplace=True)
# df = pd.concat([df, df_extended], ignore_index=True)
# df = df.sample(frac=1, random_state=42).reset_index(drop=True)

def clean_text(text):
    if not isinstance(text, str): 
        return ""
    
    text = re.sub(r'<.*?>', ' ', text)
    # text = re.sub(r'http\S+', 'httpaddr', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# df.drop_duplicates(subset=['text'], keep='first', inplace=True)
# df_ham_1000.drop_duplicates(subset=['text'], keep='first', inplace=True)
# df_new_1000.drop_duplicates(subset=['text'], keep='first', inplace=True)
# df_old.drop_duplicates(subset=['text'], keep='first', inplace=True)
# df_add_train.drop_duplicates(subset=['text'], keep='first', inplace=True)
# df_add_test.drop_duplicates(subset=['text'], keep='first', inplace=True)

# df_train, df_test = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=42)
# df_train = pd.concat([df_train, df_add_train, df_ham_1000, df_new_1000, df_old], ignore_index=True)
# df_test = pd.concat([df_test, df_add_test], ignore_index=True)
df_train = pd.read_csv('../data/train_data.csv')
df_test = pd.read_csv('../data/test_data.csv')
df_train = df_train.sample(frac=1, random_state=42).reset_index(drop=True)
df_test = df_test.sample(frac=1, random_state=42).reset_index(drop=True)

df_train['text'] = df_train['text'].apply(clean_text)
df_test['text'] = df_test['text'].apply(clean_text)

# df['label'] = df['label'].map({'Ham': 0, 'Spam': 1})
# df_extra = pd.read_csv('MASTER_SPAM_DATASET.csv')
# df = pd.concat([df, df_extra], ignore_index=True)

In [4]:
from transformers import DistilBertTokenizer
from sklearn.model_selection import train_test_split

tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-multilingual-cased')

MAX_LENGTH = 256 

def text_to_bert_sequence(text):
    if not isinstance(text, str): text = ""
    tokens = tokenizer(
        text, add_special_tokens=True, max_length=MAX_LENGTH,
        truncation=True, padding='max_length', return_tensors='np'
    )
    return tokens['input_ids'][0]

features_train = np.stack(df_train['text'].apply(text_to_bert_sequence).values)
features_test = np.stack(df_test['text'].apply(text_to_bert_sequence).values)
labels_train = df_train['label'].values
labels_test = df_test['label'].values

X_train, X_test, y_train, y_test = (features_train, features_test, labels_train, labels_test)

In [5]:
train_data = torch.utils.data.TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
test_data = torch.utils.data.TensorDataset(torch.from_numpy(X_test), torch.from_numpy(y_test))

batch_size = 128 

train_loader = DataLoader(train_data, shuffle=True, batch_size=batch_size)
test_loader = DataLoader(test_data, shuffle=False, batch_size=batch_size)

# dataiter = iter(train_loader)
# sample_x, sample_y = next(dataiter)

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_13236\269757176.py:1: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  train_data = torch.utils.data.TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))


In [6]:
from transformers import DistilBertModel
import torch.nn as nn
import torch.nn.functional as F
import torch

bert = DistilBertModel.from_pretrained('distilbert-base-multilingual-cased')

class CNN_Spam_Model(nn.Module):
    def __init__(self, n_filters, output_dim, dropout, pad_idx):
        super(CNN_Spam_Model, self).__init__()
        
        self.embedding = nn.Embedding.from_pretrained(
            bert.embeddings.word_embeddings.weight, 
            freeze=True, 
            padding_idx=pad_idx
        )
        
        embedding_dim = 768 
        
        self.conv3 = nn.Conv1d(embedding_dim, n_filters, 3)
        self.conv5 = nn.Conv1d(embedding_dim, n_filters, 5)
        self.conv7 = nn.Conv1d(embedding_dim, n_filters, 7)
        self.conv9 = nn.Conv1d(embedding_dim, n_filters, 9) 
        
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(n_filters * 4, output_dim) 
        
    def forward(self, text):
        embedded = self.embedding(text).permute(0, 2, 1) 
        
        convolved3 = F.relu(self.conv3(embedded)) 
        convolved5 = F.relu(self.conv5(embedded)) 
        convolved7 = F.relu(self.conv7(embedded)) 
        convolved9 = F.relu(self.conv9(embedded))
        
        pooled3 = F.max_pool1d(convolved3, convolved3.shape[2]).squeeze(2) 
        pooled5 = F.max_pool1d(convolved5, convolved5.shape[2]).squeeze(2) 
        pooled7 = F.max_pool1d(convolved7, convolved7.shape[2]).squeeze(2) 
        pooled9 = F.max_pool1d(convolved9, convolved9.shape[2]).squeeze(2)
        
        cat = torch.cat((pooled3, pooled5, pooled7, pooled9), dim=1) 
        return self.fc(self.dropout(cat))

PAD_IDX = tokenizer.pad_token_id
N_FILTERS = 128 
OUTPUT_DIM = 1

model = CNN_Spam_Model(n_filters=N_FILTERS, output_dim=OUTPUT_DIM, dropout=0.4, pad_idx=PAD_IDX)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-multilingual-cased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
import torch.optim.lr_scheduler as lr_scheduler

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
scheduler = lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=5, T_mult=1)
print(f"Mô hình sẽ chạy trên: {device}")

Mô hình sẽ chạy trên: cuda


In [8]:
epochs = 15

for epoch in range(epochs):
    model.train()
    total_loss = 0
    
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device).float()
        
        optimizer.zero_grad()
        
        outputs = model(batch_x).squeeze(1)
        
        loss = criterion(outputs, batch_y)
        
        loss.backward()
        
        optimizer.step()
        
        total_loss += loss.item()
    
    scheduler.step()
    # scheduler.zero_grad()
    avg_loss = total_loss / len(train_loader)
    print(f"Lần học thứ {epoch+1}/{epochs} | Độ sai số (Loss): {avg_loss:.4f}")

    model.eval() 
    correct = 0
    total = 0

    with torch.no_grad(): 
        for batch_x, batch_y in test_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device).float()
            outputs = model(batch_x).squeeze(1)
            
            predictions = (torch.sigmoid(outputs) > 0.5).float()
            
            total += batch_y.size(0)
            correct += (predictions == batch_y).sum().item()

    print(f"Độ chính xác trên tập Test lần {epoch+1}: {(100 * correct / total):.2f}%")

Lần học thứ 1/15 | Độ sai số (Loss): 0.1647
Độ chính xác trên tập Test lần 1: 97.75%
Lần học thứ 2/15 | Độ sai số (Loss): 0.0390
Độ chính xác trên tập Test lần 2: 97.94%
Lần học thứ 3/15 | Độ sai số (Loss): 0.0148
Độ chính xác trên tập Test lần 3: 98.53%
Lần học thứ 4/15 | Độ sai số (Loss): 0.0073
Độ chính xác trên tập Test lần 4: 98.70%
Lần học thứ 5/15 | Độ sai số (Loss): 0.0051
Độ chính xác trên tập Test lần 5: 98.72%
Lần học thứ 6/15 | Độ sai số (Loss): 0.0084
Độ chính xác trên tập Test lần 6: 98.66%
Lần học thứ 7/15 | Độ sai số (Loss): 0.0053
Độ chính xác trên tập Test lần 7: 98.85%
Lần học thứ 8/15 | Độ sai số (Loss): 0.0028
Độ chính xác trên tập Test lần 8: 98.82%
Lần học thứ 9/15 | Độ sai số (Loss): 0.0015
Độ chính xác trên tập Test lần 9: 98.91%
Lần học thứ 10/15 | Độ sai số (Loss): 0.0011
Độ chính xác trên tập Test lần 10: 98.88%
Lần học thứ 11/15 | Độ sai số (Loss): 0.0059
Độ chính xác trên tập Test lần 11: 98.51%
Lần học thứ 12/15 | Độ sai số (Loss): 0.0064
Độ chính xác trê

In [9]:
model.eval() 
correct = 0
total = 0

with torch.no_grad(): 
    for batch_x, batch_y in test_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device).float()
        outputs = model(batch_x).squeeze(1)
        
        predictions = (torch.sigmoid(outputs) > 0.5).float()
        
        total += batch_y.size(0)
        correct += (predictions == batch_y).sum().item()

print(f"Độ chính xác trên tập Test: {(100 * correct / total):.2f}%")

Độ chính xác trên tập Test: 98.93%


In [12]:
torch.save(model.state_dict(), 'Test_spam_cnn_model.pth')
print("Đã lưu vào file 'Test_spam_cnn_model.pth'!")

Đã lưu vào file 'Test_spam_cnn_model.pth'!
